In [14]:
import csv
import json
from collections import defaultdict
import sys

# Usage: python csv_to_network.py input.csv output.json
if len(sys.argv) < 3:
    print("Usage: python csv_to_network.py input.csv output.json")
    sys.exit(1)

infile = sys.argv[1]
outfile = sys.argv[2]

# Configure these column names to match your CSV
AUTHOR_COL = "Author"
COAUTHORS_COL = "Coauthors"     # a cell like "Bob;Charlie;Alice" or "Bob, Charlie"
AFFIL_COL = "Affiliation"
YEAR_COL = "Year"
PAPER_ID_COL = "PaperID"        # optional

def split_coauthors(cell):
    if not cell: return []
    # try common separators
    for sep in [';', ',', '|', '/']:
        if sep in cell:
            return [x.strip() for x in cell.split(sep) if x.strip()]
    return [cell.strip()]

rows = []
with open(infile, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for r in reader:
        # filter missing required fields
        if not r.get(AUTHOR_COL) or not r.get(AFFIL_COL) or not r.get(YEAR_COL):
            continue
        rows.append(r)

# Build nodes: track publications count and affiliation (if multiple affils for same author, keep first)
nodes_info = {}
# Build links by (authorA, authorB, paper) to count coauthorship weight
coauthor_pairs = defaultdict(int)

for r in rows:
    author = r[AUTHOR_COL].strip()
    aff = r[AFFIL_COL].strip()
    paper = r.get(PAPER_ID_COL) or r.get("DOI") or r.get("Title") or ""
    co_str = r.get(COAUTHORS_COL, "")
    coauthors = split_coauthors(co_str)

    # Ensure author in nodes
    if author not in nodes_info:
        nodes_info[author] = {"id": author, "affiliation": aff, "publications": 0}
    nodes_info[author]["publications"] += 1

    # Add coauthor links (author <-> each coauthor)
    for co in coauthors:
        if not co or co == author: continue
        a, b = sorted([author, co])
        key = (a, b)
        # increment by 1 for this coauthorship instance (paper)
        coauthor_pairs[key] += 1

# Convert to lists
nodes = list(nodes_info.values())
links = []
for (a,b), w in coauthor_pairs.items():
    links.append({"source": a, "target": b, "weight": w})

out = {"nodes": nodes, "links": links}
with open(outfile, "w", encoding="utf-8") as f:
    json.dump(out, f, indent=2, ensure_ascii=False)

print(f"Wrote {len(nodes)} nodes and {len(links)} links to {outfile}")


FileNotFoundError: [Errno 2] No such file or directory: '-f'

In [16]:
!python csv_to_network.py -f data_scopus.csv


python: can't open file '/Users/lavanya/Documents/Data Visualization/Major_assignment3/author-network/csv_to_network.py': [Errno 2] No such file or directory


In [5]:
import pandas as pd

df = pd.read_csv("data_scopus.csv")
print(df.columns)



Index(['Title', 'Year', 'EID', 'Abstract', 'Publisher', 'Conference name',
       'Conference date', 'Authors', 'Author(s) ID',
       'Authors with affiliations', 'Source title', 'Abbreviated Source Title',
       'Cited by'],
      dtype='object')
